<a href="https://colab.research.google.com/github/snr-laboratory/snrlab-ic-q-pix-v1/blob/main/dev_journals/kgosine_2026_02_planning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Chip Constraints



Tapeout Schedule: https://www.musesemi.com/shared-block-tapeout-schedule

11/18/26 is the last 180nm RF tapeout of the year.

Specs for 180nm: https://www.musesemi.com/shared-block-tapeout-pricing

Minimum area is 5mm^2 with aspect ration less than 3 and Y-dimension must be 3mm with X-dimension of 1.66mm as an estimate. Given the 100um pad pitch this amounts to about 30 pins along the long edge and 16 along the short edge for a total of 92 pins. Pins are expected to be the limiting factor in the design.

A single chip should have mulitple analog front ends, some kind of multiplexing to reduce pins for outputting data, and test structures.



## Detector Considerations


The minimum batch is 40 chips. If 36 could pass bench testing, this would amount to a 6 by 6 grid of chips. Assume each chip holds 9 channels (optimistic estimate), with pixels of pitch 4mm this would have a sensitive area of 51.84 cm^2 (7.2 cm by 7.2 cm). Total number of channels would be 9 by 36 = 324.

## Data Packet Construction

Each channel could have a dedicated single-ended output pin (9/92). The problem with just collecting the replenishment pulses at the FPGA is (1) the FPGA could not take in 324 signals. On the PCB there would need to be some multiplexing. (2) Timestamping the replenishments only at the FPGA would be inaccurate due to arrival time variations from chip locations.

It would therefore make more sense for data packets to be formed on chip and include a timestamp and channel id. This would require two steps:

(1) Data packet is formed for 'each' replenishement at the output of each channel including channel id, chip id, timestamp, replenishment pattern
(2) Data packet from each channel enters FIFO to be held while data from all channels exit the FIFO.

For 36 chips there need to be 6 bits for chip ID
For 9 channels we need 4 bits for channel ID
Timestamp bits ... assume 22 for now (for a 50MHz clock this resets every 82 miliseconds).
A bit indicating the start (and possibly additional bit to mark the end) of a data packet.

A data packet may contain information about a stream of replenishments. Let a single data packet contain 8 bits of information.
These 8-bit mini-packets should be formed using the following valid condition to ensure data apckets of 0s are not created and entered to the FIFO.

This is at least a 42-bit data packet per chip. If it goes out in a 16-wide bus ( all the pins on a single edge) it could be read out in 3 readout clock ticks.



![](https://drive.google.com/uc?export=view&id=1F91zB-IK8r0b3uKF7w2yKmwjZPJoG1Z5)

## Power Limitations
